Exploratory Data Analysis of Yelp Datasets

The goals of this exploratation are the following:
1) Mirror data exploration between health plans and businesses
2) Possibly scrape data by scratch from one chose health plan
3) Use sentiment analysis/NLP in order to create a metric for patient satisfaction

**Part I. Scrape Data from Health Plan on Yelp**

In [ ]:
# Import modules
from serpapi import GoogleSearch
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from os import path
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import string

from sklearn.metrics import classification_report

# NLTK
import nltk
nltk.download('stopwords')
nltk.download("vader_lexicon")
nltk.download('wordnet')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# ROBERTA
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from scipy.special import softmax

# ML (for future use)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# From Yelp Search
params = {
  "api_key": "9cb8292339aa076a0d9506cfcb304371a8aee73b5644e12f1a9221edb8b35c22",
  "engine": "yelp",
  "find_loc": "Northern California, CA",
  "find_desc": "Blue Shield of California"
}

search = GoogleSearch(params)
search_results = search.get_dict()

In [ ]:
json_version = json.dumps(search_results, indent=4)
json_dict = json.loads(json_version)

In [ ]:
# From specific place (Blue Shield California -> El Dorado Hills)

params = {
  "api_key": "9cb8292339aa076a0d9506cfcb304371a8aee73b5644e12f1a9221edb8b35c22",
  "engine": "yelp_reviews",
  "place_id": "LvpNwsSOVk9ciIHrD2NRzQ"
}

search = GoogleSearch(params)
location1_results = search.get_dict()

In [ ]:
json_version = json.dumps(location1_results, indent=4)
json_dict = json.loads(json_version)

**Part II. Exploratory Data Analysis of Review Page**

In [ ]:
# Convert JSON to Dataframe
# Extract the reviews list
reviews = json_dict['reviews']

# Normalize the review data
df = pd.json_normalize(reviews,
    meta=['position',['user', 'name'],'date',['comment', 'text'],'rating'])

In [ ]:
df.columns
# Drop unneccesary columns. We can bring these back if deemed necessary for future exploratory data analysis
# The purpose of dropping these columns is to clean up the data

df = df.drop(columns=['feedback.useful', 'feedback.funny','feedback.cool', 'tags',
                      'user.friends', 'user.photos', 'user.reviews','owner_replies','user.thumbnail', 'user.link'])

print(df)

In [ ]:
# Show average of yelp ratings

location1_mean = df['rating'].mean()
print(location1_mean)

The mean as shown above is approximately 1.58. This is close to the two star rating of Blue Shield California on the website. 
It is recommended to collect and merge data from different website pages (some insurance companies have multiple data sites on Yelp). To retain the same metric from the existing format, it  is also recommended that cutpoints are implemented (to be further explained in conclusion and recommendations)

In [ ]:
# Show distribution of ratings

# Get counts of each unique rating
rating_counts = df['rating'].value_counts()
print(rating_counts)

# Plot the bar chart
plt.bar(rating_counts.index, rating_counts.values, color='orange')
plt.xlabel('Rating')
plt.ylabel('Number of Reviews')
plt.title('Rating Distribution')
plt.show()

In [ ]:
# Show most common words

In [ ]:
# Create positive and negative category
# 1 means positive, 0 means negative
# It is positive if the rating is more than 3. If it is equal to or less than 3, it is negative 

df['category'] = np.where(df['rating'] <= 3, 0, 1)

In [ ]:
# Create word clouds

df_final = df[['user.name','comment.text', 'category']].copy()

# Filter and concatenate positive and negative comments
pos = df_final[df_final['category'] == 1]['comment.text']
pos_string = ' '.join(pos.astype(str))

neg = df_final[df_final['category'] == 0]['comment.text']
neg_string = ' '.join(neg.astype(str))

# Create a word cloud image
wc = WordCloud(background_color="white", max_words=1000, contour_width=3, contour_color='firebrick')

# Generate a POSITIVE wordcloud
wc.generate(pos_string)

# show
plt.figure(figsize=[20,10])
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.show()

In [ ]:
# Generate a NEGATIVE wordcloud
wc.generate(neg_string)

# show
plt.figure(figsize=[20,10])
plt.imshow(wc, interpolation='bilinear')
plt.axis("off")
plt.show()

**Part III. Sentiment Analysis**

In [ ]:
# Feature Engineering

# Initialize tools
sw = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Combined preprocessing function
def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in sw]
    
    return " ".join(cleaned_tokens)

# Apply to your DataFrame
df_final['comment.text'] = df_final['comment.text'].astype(str).apply(preprocess_text)

In [ ]:
# Load DF final that will be used in sentiment analysis

df_final

**Sentiment Analysis Using VADER - NLTK Sentiment Analyzer**

In [ ]:
# Initialize the NLTK analyzer
analyzer = SentimentIntensityAnalyzer()

**What is the sentiment intensity analyzer?**

The Sentiment Intensity Analyzer  is class under **VADER (Valence Aware Dictionary and Sentiment Reasoner)**, which is a rule-based sentiment analyzer that has been trained on social media text. 

The polarity score is from -1 to 1, where -1 means most negative and 1 means most positive. Sentiment labels are assigned according to the polarity score: -1 to 0.25 as negative, 0.25 to 1 as positive.

In [ ]:
def get_sentiment_scores(text):
    return analyzer.polarity_scores(text)

# Apply the function to get all VADER scores and expand into columns
df_final[['vader_neg', 'vader_neutral', 'vader_pos', 'vader_compound']] = (
    df_final['comment.text']
    .apply(get_sentiment_scores)
    .apply(pd.Series)
)

# Create binary sentiment label
def classify_sentiment(compound_score):
    if compound_score < 0.25:
        return 0
    else:
        return 1

df_final['sentiment'] = df_final['vader_compound'].apply(classify_sentiment)

df_final

In [ ]:
# Join previous dataframe with preprocessed dataframe in order to compare sentiment, category, and rating number

merged_df = pd.merge(df, df_final, on='user.name', how='outer', suffixes=('_df', '_df_final'))
merged_df = merged_df.drop(columns=['category_df','comment.text_df','vader_compound','vader_pos',
                                    'vader_neg', 'vader_neutral','comment.language'])
merged_df.reset_index(inplace=True)

merged_df

In [ ]:
# Add column that translates sentiment column to "positive" vs "negative"

merged_df['sentiment_label'] = np.where(merged_df['sentiment'] == 1, 'positive', 'negative')
merged_df

There are entries in which the sentiment analysis does not match up with the category cutpoints (1 for positive if it received more than 3 stars, 0 for negative if it received a score equal to or less than 3. Let us explore what these text entries look like.

In [ ]:
# Print the rows with mistakes
merged_df[merged_df['category_df_final'] != merged_df['sentiment']]

As observed, the sentiment analyzer concluded that the comments are more positive than negative. This contradicts the rating (rating column) which is extracted and rounded off from the website (Yelp) itself. This may be due to certain words such as "great" as seen in index number 11. 

In [ ]:
# Print the comments which generated the misaligned values
merged_df[merged_df['category_df_final'] != merged_df['sentiment']]['comment.text_df_final']

**Sentiment Analysis Using ROBERTA Model**

**What is the ROBERTA model?**

ROBERTA stands for Robustly Optimized BERT Pretraining Approach. This model is an improved version of the BERT architecture which is trained on smaller batches of data.  Specifically, it is a transformed-based language model that relies on self-attention in order to track relationships of different elements.

In [ ]:
# Intiialize the ROBERTA mo
# Load model directly
from transformers import pipeline

pipe = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment-latest")

from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-base-sentiment-latest")

**Part IV. Application of Sentiment Analysis to Medical Groups**

For this, we will be focusing on health groups which are found in the 44 counties listed within California. First, we will scrape data from Yelp websites per medical group for each county

**Possible Issues**

In applying sentiment analysis for specific and smaller scale groups, the data may vary in size. Given the limited data, the size may be too small too show significance. This calls for an emphasis on analysis that is more qualitative in nature. We will still apply the quantitative analysis through sentiment analysis, however this is not recommended. 